# PRACTICE 3: CLUSTERING RETAIL CUSTOMERS
## CHẶNG 2: TIỀN XỬ LÝ & KỸ THUẬT ĐẶC TRƯNG (DATA PREPROCESSING & FEATURE ENGINEERING)

> **Mục tiêu kỹ thuật:**
> - Làm sạch dữ liệu, chuẩn hóa tên cột tiếng Anh không dấu cách `['CustomerID', 'Gender', 'Age', 'Annual_Income_k$', 'Spending_Score']`.
> - Kiểm tra và giữ nguyên 2 quan sát ngoại lai VIP (thu nhập $137k) vì là hành vi tiêu dùng có thật.
> - Mã hóa biến danh mục Gender sang nhị phân `Gender_Code` (Female: 0, Male: 1).
> - Thiết lập 3 không gian gom cụm: 2D (Chính), 3D (Mở rộng tuổi), 4D (Toàn diện).
> - Chuẩn hóa thang đo `StandardScaler` (Mean = 0, Std = 1) bảo đảm công bằng khoảng cách Euclidean.
> - Xuất tệp dữ liệu sạch `data/processed/clean_customers.csv` phục vụ huấn luyện.

---
## Mục 1: Làm Sạch Dữ Liệu & Mã Hóa Biến Danh Mục

### 1.1 Khai Báo Thư Viện & Thiết Lập Hệ Thống
* **Mục tiêu:** Nạp các module tiền xử lý từ `data.scripts` và `scripts.feature_engineering`.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.common import RAW_CSV, PROCESSED_CSV, to_rel_path
from data.scripts.data_preprocessing import load_raw_data, clean_customer_data
from scripts.feature_engineering import encode_gender, scale_features, prepare_clustering_features

### 1.2 Thực Thi Làm Sạch & Chuẩn Hóa Tên Cột
* **Mục tiêu:** Áp dụng hàm `clean_customer_data` để làm sạch dữ liệu và kiểm định logic nghiệp vụ.

In [ ]:
df_raw = load_raw_data(RAW_CSV)
df_clean, clean_report = clean_customer_data(df_raw)

print(f"✔ Đã làm sạch dữ liệu: {clean_report['clean_shape'][0]} dòng x {clean_report['clean_shape'][1]} cột.")
print(f"• Tên các cột đã chuẩn hóa: {clean_report['columns']}")
print(f"• Số ngoại lai VIP được giữ nguyên: {clean_report['n_outliers']} khách hàng.")
display(df_clean.head())

### 1.3 Mã Hóa Giới Tính (Label Encoding Gender)
* **Mục tiêu:** Tạo biến số học `Gender_Code` (0: Female, 1: Male) phục vụ cho các thuật toán khoảng cách.

In [ ]:
df_encoded = encode_gender(df_clean)
print("Bảng tần số phân bố biến Gender_Code sau mã hóa:")
display(df_encoded[['Gender', 'Gender_Code']].value_counts().reset_index(name='Số lượng'))

---
## Mục 2: Thiết Lập Các Không Gian Phân Cụm & Chuẩn Hóa Thang Đo

### 2.1 Chuẩn Hóa Thang Đo StandardScaler
* **Mục tiêu:** Loại bỏ sự chênh lệch đơn vị giữa Thu nhập (hàng chục nghìn đô) và Điểm chi tiêu (thang 1-100).

In [ ]:
feature_cols = ['Annual_Income_k$', 'Spending_Score']
X_scaled, scaler = scale_features(df_encoded, feature_cols=feature_cols, method='standard')

print("Kiểm tra thông số thống kê của ma trận X sau khi chuẩn hóa:")
print(f"• Shape ma trận X: {X_scaled.shape}")
print(f"• Mean từng cột: {np.mean(X_scaled, axis=0).round(4)}")
print(f"• Std từng cột : {np.std(X_scaled, axis=0).round(4)}")

### 2.2 Thiết Lập 3 Không Gian Gom Cụm Linh Hoạt (2D, 3D, 4D)
* **Mục tiêu:** Tạo sẵn các ma trận đặc trưng cho các bài toán phân cụm khác nhau qua hàm `prepare_clustering_features`.

In [ ]:
X_2d, s_2d, f_2d = prepare_clustering_features(feature_set='2d')
X_3d, s_3d, f_3d = prepare_clustering_features(feature_set='3d')
X_4d, s_4d, f_4d = prepare_clustering_features(feature_set='4d')

print("1. Không gian 2D (Chính)   :", f_2d, "-> Shape:", X_2d.shape)
print("2. Không gian 3D (Mở rộng) :", f_3d, "-> Shape:", X_3d.shape)
print("3. Không gian 4D (Toàn diện):", f_4d, "-> Shape:", X_4d.shape)

### 2.3 Xuất Tệp Dữ Liệu Sạch Sẵn Sàng Huấn Luyện
* **Mục tiêu:** Lưu tệp `clean_customers.csv` vào thư mục `data/processed/`.

In [ ]:
PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
df_encoded.to_csv(PROCESSED_CSV, index=False)
print(f"✔ Đã lưu tệp dữ liệu sạch thành công tại: {to_rel_path(PROCESSED_CSV)}")
print(f"• Quy mô tệp: {df_encoded.shape[0]} dòng x {df_encoded.shape[1]} cột.")